# AIT-ADS baseline comparison

Compares six system-task baselines evaluated on one AIT-ADS scenario's train/test split -- the AIT-ADS counterpart to `notebooks/baselines/cscas_baseline_comparison.ipynb` (see `baselines/_ait_ads_data.py`, `baselines/ait_ads_rf.py`, `baselines/ait_ads_logreg.py`, `baselines/ait_ads_xgboost.py`, `baselines/ait_ads_bert.py`, `baselines/ait_ads_securebert.py`, `baselines/ait_ads_zeroshot.py`):

1. **Base schema (RF)** -- 5-feature `base` schema, `RandomForestClassifier`
2. **Base schema (LogReg)** -- same, `StandardScaler` + `LogisticRegression`
3. **Base schema (XGBoost)** -- same, `XGBClassifier`
4. **BERT** -- alert_group tokens (`sig:`/`host:`/`short:`) serialized to text, fine-tuned DistilBERT
5. **SecureBERT 2.0** -- same tokens, fine-tuned SecureBERT 2.0 (ModernBERT architecture, domain-adapted)
6. **Zero-shot** -- same tokens serialized to a prompt, no fine-tuning

**Two training-pool conditions, not three**: random undersampling and class-weighted (natural-ratio). No "guided" condition here -- AIT-ADS has no SCAS-equivalent outlier signal to guide sampling with (CSCAS-only, see `baselines/_sampling.py`'s own docstring). Zero-shot has no training step, so neither condition applies to it -- it reports a single flat precision/recall/f1 from one deterministic run.

All six methods are **seed-averaged (`N_SEEDS = 5`) except zero-shot** (single deterministic run, temperature=0), same convention as the CSCAS scripts.

**All six methods share the exact same split** -- every script calls `_ait_ads_data.load_ait_ads_baseline_split(scenario, grouping_method)`, so results are directly comparable across model families for a given `(SCENARIO, GROUPING_METHOD)`, not just similarly configured pipelines. This replaced an earlier approach that pulled RF/LogReg/XGBoost from a separate, fixed_window-only pipeline (`run_model_comparison_attribute.py`) -- that could only ever guarantee *similar* splits, not identical ones, and couldn't be extended to the other 4 grouping methods without duplicating this same split logic anyway.

Results are **per (grouping method, scenario)** -- unlike CSCAS (pregrouped), AIT-ADS alerts need a grouping step first, and which of the 5 grouping methods (`fixed_window`, `time_delta`, `cscas_grouping`, `alertbert`, `deepcase`) is used is itself an axis, not a fixed choice (see `baselines/_ait_ads_grouping.py`). Set `SCENARIO` and `GROUPING_METHOD` in the Settings cell below.

**`alertbert`/`deepcase` are only valid for `fox`, `harrison`, `russellmitchell`, `santos`** -- both the pretrained AlertBERT checkpoint and the DeepCASE ContextBuilder were trained on `shaw`/`wardbeck`/`wheeler`/`wilson`, so grouping those 4 scenarios with either method for a baseline result would be self-training leakage. Every AIT-ADS baseline script (all six model families) skips that combination automatically.

**Run the scripts first** to generate the `results/*.json` files this notebook reads:
```
cd src/thesis/baselines
python ait_ads_rf.py
python ait_ads_logreg.py
python ait_ads_xgboost.py
python ait_ads_bert.py
python ait_ads_securebert.py
OLLAMA_MODEL=llama3.1:8b python ait_ads_zeroshot.py
```


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis.baselines._results import RESULTS_DIR, is_zero_shot, load_baseline_results

## Settings

Edit `SCENARIO` (and the zero-shot model slug below, if you ran a different Ollama model) and re-run -- nothing past this cell needs to change.

In [ ]:
SCENARIO = "fox"  # fox, harrison, russellmitchell, santos, shaw, wardbeck, wheeler, wilson
GROUPING_METHOD = "fixed_window"  # fixed_window, time_delta, cscas_grouping, alertbert, deepcase
# alertbert/deepcase are only valid for fox/harrison/russellmitchell/santos -- see the intro cell above.
ZEROSHOT_MODEL_SLUG = "llama3.1-8b"  # matches OLLAMA_MODEL with ":"/"/" replaced by "-", see ait_ads_zeroshot.py

RESULT_NAMES = [
    f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}",
    f"ait_ads_zeroshot_{GROUPING_METHOD}_{SCENARIO}_{ZEROSHOT_MODEL_SLUG}",
]

LABELS = {
    f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}": f"Base schema (RF, {GROUPING_METHOD})",
    f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}": f"Base schema (LogReg, {GROUPING_METHOD})",
    f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}": f"Base schema (XGBoost, {GROUPING_METHOD})",
    f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}": f"BERT (tokens as text, {GROUPING_METHOD})",
    f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}": f"SecureBERT 2.0 (tokens as text, {GROUPING_METHOD})",
    f"ait_ads_zeroshot_{GROUPING_METHOD}_{SCENARIO}_{ZEROSHOT_MODEL_SLUG}": f"Zero-shot ({ZEROSHOT_MODEL_SLUG}, {GROUPING_METHOD})",
}

COLORS = {
    f"ait_ads_rf_{GROUPING_METHOD}_{SCENARIO}": "#CCBB44",
    f"ait_ads_logreg_{GROUPING_METHOD}_{SCENARIO}": "#228833",
    f"ait_ads_xgboost_{GROUPING_METHOD}_{SCENARIO}": "#66CCEE",
    f"ait_ads_bert_{GROUPING_METHOD}_{SCENARIO}": "#EE6677",
    f"ait_ads_securebert_{GROUPING_METHOD}_{SCENARIO}": "#AA3377",
    f"ait_ads_zeroshot_{GROUPING_METHOD}_{SCENARIO}_{ZEROSHOT_MODEL_SLUG}": "#BBBBBB",
}

# Scoped per (scenario, grouping method) so different combinations' figures/summaries don't overwrite each other.
FIGURES_DIR = RESULTS_DIR / "figures" / SCENARIO / GROUPING_METHOD
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
results = {}
for name in RESULT_NAMES:
    try:
        results[name] = load_baseline_results(name)
    except FileNotFoundError as e:
        print(f"[skip] {e}")

print(f"Loaded results for: {list(results.keys())}")

In [ ]:
def plot_baseline(condition_key: str, title: str) -> None:
    """Grouped bar chart across every loaded method that actually reports
    `condition_key` -- excludes zero-shot always (flat shape, no
    conditions -- see plot_zeroshot() below). Saves to
    FIGURES_DIR/{condition_key}.png before displaying."""
    metrics = ["precision", "recall", "f1"]
    methods = [
        n for n in RESULT_NAMES
        if n in results and not is_zero_shot(results[n]) and condition_key in results[n]
    ]
    if not methods:
        print(f"No results loaded with a '{condition_key}' condition -- run the baseline scripts first.")
        return

    x = np.arange(len(metrics))
    width = 0.85 / len(methods)

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, name in enumerate(methods):
        values = [results[name][condition_key][m] for m in metrics]
        offset = (i - (len(methods) - 1) / 2) * width
        bars = ax.bar(
            x + offset, values, width,
            label=LABELS.get(name, name), color=COLORS.get(name),
        )
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1.2)
    ax.set_ylabel("Score")
    ax.set_title(f"{title} ({SCENARIO}, {GROUPING_METHOD})")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    save_path = FIGURES_DIR / f"{condition_key}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Figure written to {save_path}")

    plt.show()

## Random undersampling

In [ ]:
plot_baseline("random", "Random undersampling")

## Class-weighted (natural-ratio)

In [ ]:
plot_baseline("class_weighted", "Class-weighted (natural-ratio)")

## Zero-shot

The one method without the two-condition structure (no training pool, no seeds) -- a single bar per metric from one deterministic run, not comparable to `plot_baseline()`'s per-condition grouped bars above.

In [ ]:
def plot_zeroshot() -> None:
    name = next((n for n in RESULT_NAMES if n in results and is_zero_shot(results[n])), None)
    if name is None:
        print("No zero-shot results loaded -- run ait_ads_zeroshot.py first (check ZEROSHOT_MODEL_SLUG above matches the model you ran).")
        return

    metrics = ["precision", "recall", "f1"]
    data = results[name]
    values = [data[m] for m in metrics]

    fig, ax = plt.subplots(figsize=(4, 4))
    bars = ax.bar(metrics, values, color=COLORS.get(name))
    ax.bar_label(bars, fmt="%.3f", fontsize=9, padding=2)

    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(f"{LABELS.get(name, name)} ({SCENARIO}, {GROUPING_METHOD})")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    save_path = FIGURES_DIR / "zeroshot.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Figure written to {save_path}")

    plt.show()


plot_zeroshot()

## Summary table

Tidy view of all loaded results for this scenario -- handy to copy straight into a slide.

In [ ]:
CONDITION_LABELS = {
    "random": "Random undersampling",
    "class_weighted": "Class-weighted (natural-ratio)",
}

rows = []
for name, data in results.items():
    if is_zero_shot(data):
        rows.append({
            "method": LABELS.get(name, name),
            "condition": "(no training)",
            "precision": data["precision"],
            "recall": data["recall"],
            "f1": data["f1"],
        })
    else:
        for condition_key, condition_label in CONDITION_LABELS.items():
            if condition_key not in data:
                continue
            rows.append({
                "method": LABELS.get(name, name),
                "condition": condition_label,
                "precision": data[condition_key]["precision"],
                "recall": data[condition_key]["recall"],
                "f1": data[condition_key]["f1"],
            })

if not rows:
    print("No results loaded -- run the baseline scripts first.")
    summary_df = pd.DataFrame(columns=["method", "condition", "precision", "recall", "f1"])
else:
    summary_df = pd.DataFrame(rows).sort_values(["condition", "method"]).reset_index(drop=True)
    summary_csv_path = RESULTS_DIR / f"ait_ads_{SCENARIO}_{GROUPING_METHOD}_baseline_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False)
    print(f"Summary written to {summary_csv_path}")
summary_df